In [4]:
# %% 两-split + bootstrap + α-grid + R-replication SVGP 校准

import math, time
from contextlib import ExitStack
from dataclasses import dataclass

import torch, gpytorch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from IPython.display import display

print("PyTorch version:", torch.__version__)
print("GPyTorch version:", gpytorch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

# ---------- 小工具 ----------

def now(): return time.strftime("%H:%M:%S")
def log(msg): print(f"[{now()}] {msg}", flush=True)
def gpu_mem_mb():
    return (torch.cuda.memory_allocated()/1e6) if torch.cuda.is_available() else 0.0

def has_setting(name: str) -> bool:
    return hasattr(gpytorch.settings, name)

def ciq_context():
    """可选 CIQ 设置（和你 baseline 一样），如果不用 CIQ 就用 ExitStack()."""
    stack = ExitStack()
    for name, val in [
        ("num_contour_quadrature", 15),
        ("ciq_samples", 16),
        ("ciq_max_iter", 30),
        ("ciq_forward_precision", 0.01),
        ("eval_cg_tolerance", 1e-4),
        ("max_root_decomposition_size", 64),
    ]:
        if has_setting(name):
            stack.enter_context(getattr(gpytorch.settings, name)(val))
    stack.enter_context(gpytorch.settings.fast_pred_var(True))
    return stack

def clone_state_dict(sd):
    out = {}
    for k, v in sd.items():
        out[k] = v.detach().clone() if torch.is_tensor(v) else v
    return out

# 真正的 latent f0（无噪音）
def f0(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * math.pi * x) + 0.5 * torch.cos(3 * x)

# 1D 分层采样（LHS-like）
def make_stratified_points_1d(n, low, high, generator, device, dtype):
    i = torch.arange(n, device=device, dtype=dtype)
    u = (i + torch.rand(n, generator=generator, device=device, dtype=dtype)) / n
    x = low + (high - low) * u
    perm = torch.randperm(n, generator=generator, device=device)
    return x[perm].unsqueeze(-1)

# ---------- 配置 ----------

@dataclass
class CFG:
    # data
    n_train: int = 500
    n_test: int = 50
    train_low: float = -2.5
    train_high: float = 2.5
    test_low: float = -2.5
    test_high: float = 2.5

    # model / training
    use_fp64: bool = True
    use_ciq: bool = False
    inducing_points: int = 50
    stage1_epochs: int = 2000     # 本地先可以改小一点比如 800
    stage2_epochs: int = 800      # 同上
    batch_size: int = 500
    lr_adam_stage1: float = 2e-2
    lr_adam_stage2: float = 2e-2
    jitter: float = 1e-2
    seed: int = 13

    # early stopping for Stage-1
    early_stop_patience: int = 60
    early_stop_min_delta: float = 1e-4

    # experiment
    alphas: tuple = (1.0, 0.1)
    replications: int = 3        # 本地先小一点，跑通后自己放大
    B_bootstrap: int = 20        # 同上
    ci_level: float = 0.95

    # logging / plots
    log_epoch: bool = False
    log_every: int = 10
    show_plots: bool = False
    save_plots: bool = True

# ---------- SVGP 模型 + 训练 helper ----------

class SVGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points_std: torch.Tensor):
        M = inducing_points_std.size(0)
        q_dist = gpytorch.variational.CholeskyVariationalDistribution(M)
        q_strat = gpytorch.variational.VariationalStrategy(
            self, inducing_points_std, q_dist, learn_inducing_locations=False
        )
        super().__init__(q_strat)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x_std):
        mean_x = self.mean_module(x_std)
        covar_x = self.covar_module(x_std)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

def freeze_all_params(model, likelihood):
    for p in model.parameters():
        p.requires_grad_(False)
    for p in likelihood.parameters():
        p.requires_grad_(False)

def pick_covariance_params_svgp(model):
    """只选 variational covariance 相关参数（排除 mean），用于 Stage-2。"""
    trainables, names = [], []
    vdist_mod = None
    vs = getattr(model, "variational_strategy", None)
    if vs is not None:
        if hasattr(vs, "_variational_distribution"):
            vdist_mod = getattr(vs, "_variational_distribution")
        else:
            cand = getattr(vs, "variational_distribution", None)
            if isinstance(cand, torch.nn.Module):
                vdist_mod = cand
    if isinstance(vdist_mod, torch.nn.Module):
        for n, p in vdist_mod.named_parameters(recurse=True):
            low = n.lower()
            if "mean" in low:
                p.requires_grad_(False)
            elif any(k in low for k in ["chol", "std", "var", "covar", "factor"]):
                p.requires_grad_(True); trainables.append(p); names.append(f"vdist.{n}")
            else:
                p.requires_grad_(False)
    if not trainables:
        allow = (
            "chol_variational_covar","variational_distribution.chol_covar",
            "variational_distribution._cholesky_factor","_variational_distribution.chol_covar",
            "raw_covar","chol_factor","covar_factor","variational_stddev","raw_var",
            "qchol","q_scale_tril"
        )
        for n, p in model.named_parameters():
            low = n.lower()
            if "variational_strategy" in low and any(k in low for k in allow) and "mean" not in low:
                p.requires_grad_(True); trainables.append(p); names.append(n)
    return trainables, names

def stage1_train(model, likelihood, x_train_std, y_train, lr, epochs, jitter, batch_size,
                 early_stop_patience, early_stop_min_delta, log_epoch=False, log_every=10):
    N = x_train_std.size(0)
    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=N, beta=1.0)

    likelihood.noise_covar.raw_noise.requires_grad_(False)
    opt = torch.optim.Adam([
        {"params": model.variational_parameters()},
        {"params": model.hyperparameters()},
        {"params": likelihood.parameters()},
    ], lr=lr)

    loader = DataLoader(TensorDataset(x_train_std, y_train),
                        batch_size=batch_size, shuffle=True, drop_last=False)

    model.train(); likelihood.train()
    neg_elbo_hist, best_loss, best_epoch = [], float("inf"), 0
    best_model_sd = clone_state_dict(model.state_dict())
    best_lik_sd   = clone_state_dict(likelihood.state_dict())
    patience = 0

    with gpytorch.settings.cholesky_jitter(jitter):
        for ep in range(1, epochs + 1):
            total, n_batches = 0.0, 0
            for xb, yb in loader:
                opt.zero_grad(set_to_none=True)
                out = model(xb)
                loss = -mll(out, yb)
                if loss.numel() > 1:
                    loss = loss.mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
                total += float(loss.item()); n_batches += 1
            avg_loss = total / max(1, n_batches)
            neg_elbo_hist.append(avg_loss)

            if best_loss - avg_loss > early_stop_min_delta:
                best_loss, best_epoch = avg_loss, ep
                best_model_sd = clone_state_dict(model.state_dict())
                best_lik_sd   = clone_state_dict(likelihood.state_dict())
                patience = 0
            else:
                patience += 1

            if log_epoch and (ep % log_every == 0 or ep == 1 or ep == epochs):
                log(f"[S1] epoch {ep:04d}/{epochs} | -ELBO={avg_loss:.6f} (best {best_loss:.6f}@{best_epoch})")

            if patience >= early_stop_patience:
                break

    model.load_state_dict(best_model_sd)
    likelihood.load_state_dict(best_lik_sd)
    model.eval(); likelihood.eval()
    return neg_elbo_hist, best_epoch, best_loss

def stage2_train_cov_only(model, likelihood, x_train_std, y_train, beta, lr, epochs, jitter, batch_size):
    N = x_train_std.size(0)
    freeze_all_params(model, likelihood)
    cov_params, cov_names = pick_covariance_params_svgp(model)
    assert cov_params, "No covariance params found to train in Stage-2."
    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=N, beta=beta)
    opt = torch.optim.Adam(cov_params, lr=lr)

    loader = DataLoader(TensorDataset(x_train_std, y_train),
                        batch_size=batch_size, shuffle=True, drop_last=False)

    model.train(); likelihood.train()
    neg_elbo_hist, best_loss, best_epoch = [], float("inf"), 0
    best_model_sd = clone_state_dict(model.state_dict())
    best_lik_sd   = clone_state_dict(likelihood.state_dict())

    with gpytorch.settings.cholesky_jitter(jitter):
        for ep in range(1, epochs + 1):
            total, n_batches = 0.0, 0
            for xb, yb in loader:
                opt.zero_grad(set_to_none=True)
                out = model(xb)
                loss = -mll(out, yb)
                if loss.numel() > 1:
                    loss = loss.mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(cov_params, 5.0)
                opt.step()
                total += float(loss.item()); n_batches += 1
            avg_loss = total / max(1, n_batches)
            neg_elbo_hist.append(avg_loss)
            if avg_loss < best_loss:
                best_loss, best_epoch = avg_loss, ep
                best_model_sd = clone_state_dict(model.state_dict())
                best_lik_sd   = clone_state_dict(likelihood.state_dict())

    model.load_state_dict(best_model_sd)
    likelihood.load_state_dict(best_lik_sd)
    model.eval(); likelihood.eval()
    return neg_elbo_hist, best_epoch, best_loss, cov_names

# ---------- 主函数：two-split bootstrap calibration ----------

def run_two_split_bootstrap_calibration(cfg: CFG):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.set_default_dtype(torch.float64 if cfg.use_fp64 else torch.float32)

    torch.manual_seed(cfg.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg.seed)

    log(f"Device: {device} | dtype: {torch.get_default_dtype()}")
    if torch.cuda.is_available():
        log(f"CUDA mem (start): {gpu_mem_mb():.0f} MB")

    # ---- 全局固定 train/test ----
    gen = torch.Generator(device=device).manual_seed(cfg.seed)
    x_train_raw = make_stratified_points_1d(
        n=cfg.n_train, low=cfg.train_low, high=cfg.train_high,
        generator=gen, device=device, dtype=torch.get_default_dtype()
    )
    x_test_raw = torch.linspace(cfg.test_low, cfg.test_high, cfg.n_test,
                                device=device, dtype=torch.get_default_dtype()).unsqueeze(1)
    y_train_full = f0(x_train_raw).squeeze(-1)
    y_test_true = f0(x_test_raw).squeeze(-1)

    # 对称线性归一化
    mid = torch.tensor([(cfg.train_low + cfg.train_high)/2.0],
                       device=device, dtype=torch.get_default_dtype())
    half = torch.tensor([(cfg.train_high - cfg.train_low)/2.0],
                        device=device, dtype=torch.get_default_dtype()).clamp_min(1e-12)
    def to_std(x_raw): return (x_raw - mid) / half

    x_train_std_full = to_std(x_train_raw)
    x_test_std  = to_std(x_test_raw)

    # 固定 Z
    Z_raw = torch.linspace(cfg.train_low, cfg.train_high, cfg.inducing_points,
                           device=device, dtype=torch.get_default_dtype()).unsqueeze(1)
    Z_std = to_std(Z_raw)

    left_std = float(to_std(torch.tensor([[cfg.train_low]], device=device)))
    right_std = float(to_std(torch.tensor([[cfg.train_high]], device=device)))
    log(f"Symmetric norm: raw [{cfg.train_low:.2f},{cfg.train_high:.2f}] -> std [{left_std:.2f},{right_std:.2f}] (≈ -1,+1)")
    log(f"Z_std head/tail: {float(Z_std[0]):.3f}, {float(Z_std[-1]):.3f}")

    # ---- 存储：inclusion indicators (A,R,B,J) ----
    alphas = list(cfg.alphas)
    A, R, J = len(alphas), cfg.replications, cfg.n_test
    B = cfg.B_bootstrap
    indicators = np.zeros((A, R, B, J), dtype=np.int8)

    z_ci = 1.96  # 对应 95% CI

    for r in range(R):
        log(f"\n===== Replication {r+1:03d}/{R} =====")
        n_train = cfg.n_train
        assert n_train % 2 == 0, "n_train must be even for equal split."
        perm = torch.randperm(n_train, device=device)
        half_n = n_train // 2
        split1_idx = perm[:half_n]
        split2_idx = perm[half_n:]

        x_s1_std = x_train_std_full[split1_idx]
        y_s1     = y_train_full[split1_idx]
        x_s2_std = x_train_std_full[split2_idx]
        y_s2     = y_train_full[split2_idx]

        # ---- Split 1: Stage-1 (alpha=1) 用来产生 pseudo-truth ----
        model_s1 = SVGPModel(Z_std.clone().contiguous()).to(device=device)
        lik_s1   = gpytorch.likelihoods.GaussianLikelihood().to(device=device)
        with torch.no_grad():
            model_s1.covar_module.base_kernel.lengthscale.fill_(1.0)
            model_s1.covar_module.outputscale.fill_(1.0)
            lik_s1.noise = torch.as_tensor(3e-3, dtype=torch.get_default_dtype(), device=device)
        lik_s1.noise_covar.raw_noise.requires_grad_(False)

        with (ciq_context() if cfg.use_ciq else ExitStack()):
            _, best_epoch_s1, best_loss_s1 = stage1_train(
                model_s1, lik_s1, x_s1_std, y_s1,
                lr=cfg.lr_adam_stage1, epochs=cfg.stage1_epochs, jitter=cfg.jitter,
                batch_size=cfg.batch_size,
                early_stop_patience=cfg.early_stop_patience,
                early_stop_min_delta=cfg.early_stop_min_delta,
                log_epoch=cfg.log_epoch, log_every=cfg.log_every
            )

        with torch.no_grad():
            pred_truth = model_s1(x_test_std)
            mu_truth = pred_truth.mean.detach()    # [J]
        log(f"[rep {r+1:03d}] Split1 Stage-1 done | best -ELBO={best_loss_s1:.6f}@{best_epoch_s1}")

        # ---- Split 2: Stage-1 (alpha=1) 作为 bootstrap 的 base model ----
        model_s2 = SVGPModel(Z_std.clone().contiguous()).to(device=device)
        lik_s2   = gpytorch.likelihoods.GaussianLikelihood().to(device=device)
        with torch.no_grad():
            model_s2.covar_module.base_kernel.lengthscale.fill_(1.0)
            model_s2.covar_module.outputscale.fill_(1.0)
            lik_s2.noise = torch.as_tensor(3e-3, dtype=torch.get_default_dtype(), device=device)
        lik_s2.noise_covar.raw_noise.requires_grad_(False)

        with (ciq_context() if cfg.use_ciq else ExitStack()):
            _, best_epoch_s2_base, best_loss_s2_base = stage1_train(
                model_s2, lik_s2, x_s2_std, y_s2,
                lr=cfg.lr_adam_stage1, epochs=cfg.stage1_epochs, jitter=cfg.jitter,
                batch_size=cfg.batch_size,
                early_stop_patience=cfg.early_stop_patience,
                early_stop_min_delta=cfg.early_stop_min_delta,
                log_epoch=cfg.log_epoch, log_every=cfg.log_every
            )
        base_model_s2_sd = clone_state_dict(model_s2.state_dict())
        base_lik_s2_sd   = clone_state_dict(lik_s2.state_dict())
        log(f"[rep {r+1:03d}] Split2 Stage-1 base done | best -ELBO={best_loss_s2_base:.6f}@{best_epoch_s2_base}")

        # ---- 对每个 alpha 做 bootstrap ----
        n_s2 = x_s2_std.size(0)
        for a_idx, alpha in enumerate(alphas):
            beta = 1.0 / alpha
            log(f"[rep {r+1:03d}]   Alpha {alpha:.3f}: bootstrap B={B}")

            for b in range(B):
                # bootstrap indices from Split2
                boot_idx = torch.randint(low=0, high=n_s2, size=(n_s2,), device=device)
                x_boot_std = x_s2_std[boot_idx]
                y_boot     = y_s2[boot_idx]

                # reset model to Stage-1 base
                model_s2.load_state_dict(base_model_s2_sd)
                lik_s2.load_state_dict(base_lik_s2_sd)
                model_s2.eval(); lik_s2.eval()

                # Stage-2 (covariance only) on bootstrap data
                with (ciq_context() if cfg.use_ciq else ExitStack()):
                    _, _, _, _ = stage2_train_cov_only(
                        model_s2, lik_s2, x_boot_std, y_boot,
                        beta=beta, lr=cfg.lr_adam_stage2, epochs=cfg.stage2_epochs,
                        jitter=cfg.jitter, batch_size=cfg.batch_size
                    )

                # predict at test grid, form CI, store inclusion indicator
                with torch.no_grad():
                    pred_b = model_s2(x_test_std)
                    mu_b   = pred_b.mean
                    std_b  = pred_b.variance.sqrt().clamp_min(1e-12)
                    lower  = mu_b - z_ci * std_b
                    upper  = mu_b + z_ci * std_b
                    inc    = ((mu_truth >= lower) & (mu_truth <= upper)).to(torch.int8)  # [J]

                indicators[a_idx, r, b, :] = inc.cpu().numpy()

                if (b+1) % max(1, (B//5)) == 0:
                    log(f"        [alpha={alpha:.3f}] bootstrap {b+1}/{B}")

    # ===== 结束后：从 indicators 计算 coverage & 生成表 =====
    x_axis = x_test_raw.squeeze(-1).cpu().numpy()   # [J]

    # 先对 B (bootstrap) 平均 -> (A,R,J)
    coverage = indicators.mean(axis=2)              # float64
    # 再对 R (replications) 平均 -> (A,J)
    coverage_mean_R = coverage.mean(axis=1)         # [A,J]
    overall_cov = coverage_mean_R.mean(axis=1)      # [A]

    # 画 coverage 曲线（可选）
    plt.figure(figsize=(7.8, 4.4))
    for a_idx, alpha in enumerate(alphas):
        plt.plot(x_axis, coverage_mean_R[a_idx],
                 label=f"α={alpha:.3f} (mean={overall_cov[a_idx]:.3f})")
    plt.hlines(cfg.ci_level, x_axis.min(), x_axis.max(),
               linestyles="--", label=f"Nominal {cfg.ci_level:.2f}")
    plt.ylim(0.0, 1.02)
    plt.title(f"Two-split bootstrap calibration | R={R}, B={B}")
    plt.xlabel("x"); plt.ylabel("Pointwise coverage (mean over reps)")
    plt.legend(); plt.tight_layout()
    if cfg.save_plots:
        plt.savefig(f"calib_two_split_coverage_R{R}_B{B}.png", dpi=150)
    if cfg.show_plots:
        plt.show()
    else:
        plt.close()

    # 保存 indicators 和 coverage
    np.save(f"indicators_array_R{R}_B{B}.npy", indicators)
    np.save(f"coverage_array_R{R}_B{B}.npy", coverage)
    log("Saved indicators and coverage arrays.")

#所有 alpha 各一列
    data_all = {"x": x_axis}
    for a_idx, alpha in enumerate(alphas):
        col_name = f"alpha_{alpha:.3f}"
        data_all[col_name] = coverage_mean_R[a_idx]
    df_all = pd.DataFrame(data_all)
    display(df_all.head())
    df_all.to_csv(f"pointwise_coverage_all_alphas_R{R}_B{B}.csv", index=False)

    log("Saved pointwise coverage tables (single alpha & all alphas).")
    return indicators, coverage, df_single, df_all

# ---- 在 notebook 里直接调用 ----
cfg = CFG()
indicators, coverage, df_single, df_all = run_two_split_bootstrap_calibration(cfg)


PyTorch version: 2.5.1+cu121
GPyTorch version: 1.14
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4080
[23:13:06] Device: cuda | dtype: torch.float64
[23:13:06] CUDA mem (start): 18 MB
[23:13:06] Symmetric norm: raw [-2.50,2.50] -> std [-1.00,1.00] (≈ -1,+1)
[23:13:06] Z_std head/tail: -1.000, 1.000
[23:13:06] 
===== Replication 001/3 =====


KeyboardInterrupt: 